In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import beatport
mio   = beatport.MusicDBIO(verbose=False)
webio = beatport.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Beatport DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Beatport


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:   {0}".format(len(localArtists.get())))
print("   Master Artists:  {0}".format(len(masterArtists.get())))
print("   Errors:          {0}".format(len(errors.get())))
print("   Search Artists:  {0}".format(searchArtists.shape[0]))
print("   Known IDs:       {0}".format(len(knownArtists)))

Beatport Search Results
   Local Artists:   116776
   Master Artists:  0
   Errors:          29
   Search Artists:  203405
   Known IDs:       88163


# Download Artist Data

In [6]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [23]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  82148
#   Artist Names To Get:  59348
#   Artist Names To Get:  45098
#   Artist Names To Get:  39998

Beatport Search Results
   Available Names:      203405
   Known Artist Names:   163401
   Artist Names To Get:  39998


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "10:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "10:00am")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Beatport ArtistIDs] Start    ==> Time Is 2022-04-23 22:23:50
========================= termTime(day=tomorrow,time=10:50am) =========================
   ====> Terminate Time Set To 2022-04-24 10:50:00 <====
   ====> Will Terminate Process 12 Hours and 26 Minutes From Now
Getting Data For Kotei (524678)                                    True
Getting Data For Jawside (524665)                                  True
Getting Data For Yosoy (524647)                                    True
Getting Data For Exem (52463)                                      True
Getting Data For Kruder & Dorfmeister (52461)                      True
Getting Data For Silver T (524599)                                 True
Getting Data For Mourad (21311)                                    True
Getting Data For Msizo (524426)                                    True
Getting Data For Silverzone (525583)                               True
Getting Data For Marco Coccia (524122)                          

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For The Rock Solid All-Stars (37964)                  True
Getting Data For Beldina (170578)                                  True
Getting Data For Beatniqz (379577)                                 True
Getting Data For Muckypuh (524403)                                 True
Getting Data For Loochkin (524340)                                 True
Getting Data For Red & Black (524384)                              True
Getting Data For R.H.X (524380)                                    True
Getting Data For 80xx (524379)                                     True
Getting Data For Clouds Round The Moon (524366)                    True
Getting Data For Far-K (524365)                                    True
Getting Data For Hissachi (524362)                                 True
Getting Data For Marubini Musiq (379517)                           True
Getting Data For Joshua Corallini (524335)                         True
Getting Data For Shamwey (524223)                               

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Buurman (525320)                                  True
Getting Data For Harmil (379126)                                   True
Getting Data For Ryeler (379127)                                   True
Getting Data For Daniele Lia (525296)                              True
Getting Data For Aurelio Guima (379142)                            True
Getting Data For Regina Jhey (379145)                              True
Getting Data For Impera (525262)                                   True
Getting Data For Trimbeat (525236)                                 True
Getting Data For Vadness (525333)                                  True
Getting Data For DJ Nylezzz (52523)                                True
Getting Data For Jeaf Gills (525228)                               True
Getting Data For Andy Barnett (525214)                             True
Getting Data For V.G (525198)                                      True
Getting Data For Tekzotic (170333)                              

...

Getting Data For Activ-8 (544523)                                  True
Getting Data For Goombay Dance Band (544519)                       True
Getting Data For Crossman (544329)                                 True
Getting Data For J. Hash (165006)                                  True
Getting Data For Gerald Brown of Shalamar (373166)                 True
Getting Data For Ting (373128)                                     True
Getting Data For Fix The Bit (373122)                              True
Getting Data For Thony Ritz (54409)                                True
Getting Data For Raul Ortiz (54408)                                True
Getting Data For Zaka Pulco (544079)                               True
Getting Data For Joker Starr (165116)                              True
Getting Data For Gabriel Uribe (544035)                            True
Getting Data For O'Harriedge (544012)                              True
Getting Data For dootdoot (544002)                              

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Ben Bright (544269)                               True
Getting Data For MC Boogshe (214931)                               True
Getting Data For Rocket Jumper (373052)                            True
Getting Data For Peppe La Rosa (544305)                            True
Getting Data For Klient (373053)                                   True
Getting Data For Atonal (544298)                                   True
Getting Data For Marty Cherry (544284)                             True
Getting Data For Housebatze (165016)                               True
Getting Data For Ramos (54426)                                     True
Getting Data For Marina Rosenfeld (165089)                         True
Getting Data For Okta (165019)                                     True
Getting Data For Alexander O'Neal (373093)                         True
Getting Data For DJ RFX (Italy) (165020)                           True
Getting Data For Krommerz (165025)                              

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Data For Hotmân (545049)                                   True
Getting Data For Joey Viper (215060)                               True
Getting Data For Lago$ (545045)                                    True
Getting Data For Sebastien Fauvel (545027)                         True
Getting Data For M3ssiah (545111)                                  True
Getting Data For Latona (372785)                                   True
Getting Data For Personality Disorder (545016)                     True
Getting Data For Deejay Sat (545012)                               True
Getting Data For Tok Tok (54501)                                   True
Getting Data For Chris Sonaxx (54500)                              True
Getting Data For Blvkstn (544998)                                  True
Getting Data For Athom (544993)                                    True
Getting Data For Mariana M (372758)                                True
Getting Data For Jag Bentley (545116)                           

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
from lib.beatport import moveLocalFiles
moveLocalFiles()

In [ ]:
from utils import PoolIO
pio = PoolIO("Beatport")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p"):
        bsdata = hio.get(io.get(ifile))
        refs  += [(ref.attrs['data-artist'],ref.attrs['href'],ref.text) for ref in bsdata.findAll('a') if ref.attrs.get('data-artist') is not None]
    if (modVal+1) % 5 == 0:
        print(modVal,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df[0].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df[0]
df.index.name = None
df = df.drop([0], axis=1)
df.columns = ["Ref", "Name"]
df = df.join(N)
df.sort_values(by='Num', ascending=False)

In [ ]:
mio.data.saveSearchArtistData(data=df)